In [1]:
import re
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoProcessor
from datasets import load_dataset
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch
from qwen_vl_utils import process_vision_info
import torch
from transformers import AutoTokenizer, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from datasets import load_dataset

In [2]:
import torch.nn as nn
import torch

class RewardModel(nn.Module):
    def __init__(self, qwen_model):
        super().__init__()
        self.backbone = qwen_model
        self.backbone.requires_grad_(False)  # 冻结主模型权重

        # Reward head：从最后一个 token 的 hidden state 映射到 [0, 1] 分数
        self.reward_head = nn.Sequential(
            nn.Linear(self.backbone.config.hidden_size, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),  # 输出一个分数
            nn.Sigmoid()  # 输出是 0~1 的 score
        )
        self.reward_head = self.reward_head.to(next(qwen_model.parameters()).dtype)

    def forward(self, input_ids, attention_mask, pixel_values, image_grid_thw):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=image_grid_thw,  # 必传且不可为 None
            output_hidden_states=True,
            return_dict=True,
        )
        last_hidden = outputs.hidden_states[-1]  # [B, S, H]
        pooled = last_hidden[:, -1, :]  # [B, H]
        score = self.reward_head(pooled).squeeze(-1)  # [B]
        
        return score

In [3]:
# 自动使用半精度和设备映射
base_model = AutoModelForVision2Seq.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)
# 加载 LoRA 微调权重（路径根据你保存的checkpoint设置）
base_model = PeftModel.from_pretrained(
    base_model,
    "/root/IC_MLLM_VQA/ScienceQA/Lora/qwen2.5vl-Lora1-6_0609/checkpoint-5000"
)
# 构建带有 reward head 的模型
rm_model = RewardModel(base_model)

# 加载你训练保存的 reward head 权重
reward_head_path = "reward_head_only.pt"
rm_model.reward_head.load_state_dict(torch.load(reward_head_path))

# 推理阶段使用 eval 模式 + 放到 CUDA
rm_model = rm_model.to("cuda").eval()



Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

/tmp/ipykernel_26408/331917216.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rm_model.reward_head.load_state_dict(torch.load(reward_head_path))


In [4]:
from PIL import Image
from tqdm import tqdm
from transformers import AutoTokenizer
import numpy as np
from torchvision import transforms

def build_filtered_dataset(dataset_name='derek-thomas/ScienceQA',
                           split='train',
                           keep_grades='1-6'):
    """
    构建按年级和图像存在性过滤的数据集。

    参数:
        dataset_name (str): 数据集名称，例如 'derek-thomas/ScienceQA'。
        split (str): 数据分割，例如 'train', 'test', 'validation'。
        keep_grades (str or None): 筛选的年级段："1-6"、"7-12" 或 None 表示不过滤。

    返回:
        List[Dict]: 筛选后的样本列表。
    """

    def is_grade_allowed(grade_str):
        if keep_grades is None:
            return True
        try:
            grade_num = int(grade_str.replace("grade", ""))
            if keep_grades == "1-6":
                return 1 <= grade_num <= 6
            elif keep_grades == "7-12":
                return 7 <= grade_num <= 12
        except:
            return False
        return False



    data = load_dataset(dataset_name, split=split)
    dataset = []

    for i, sample in enumerate(data):
        try:
            if sample.get('question') is None:
                continue
            
            if sample.get("image", None) is None:
                continue

            if not is_grade_allowed(sample.get("grade", "")):
                continue

            solution = sample.get("solution", "")
            lecture = sample.get("lecture", "")
            solution_lecture = f"{solution}\n\n{lecture}".strip()
            
            image = sample["image"].convert("RGB")
            

            # image = np.array(image)
            # image = torch.tensor(image).permute(2, 0, 1)  # shape: (C, H, W)
            dataset.append({
                "image": image, 
                "question": sample["question"],
                "choices": sample["choices"],
                "hint": sample["hint"],
                "answer": sample["answer"],
                "solution_lecture": solution_lecture,
                'grade':sample["grade"],
            })
            
        except Exception as e:
            print(f"跳过第 {i} 个样本，错误：{e}")
            continue
    return dataset

data = build_filtered_dataset(split='test', keep_grades='1-6')
print(f"\n✅ 筛选后的样本数量: {len(data)}")


✅ 筛选后的样本数量: 1429


In [6]:
import json
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# 构造消息
def build_message(sample):
    content = []
    if sample['image'] is not None:
        content.append({"type": "image", "image": sample['image']})
    
    question_text = f"Question: {sample['question']}\nChoices:\n"
    for idx, choice in enumerate(sample['choices']):
        question_text += f"{chr(65 + idx)}. {choice}\n"
    
    if sample.get("hint"):
        question_text += f"\nHint: {sample['hint']}\n"
    
    question_text += """
    Please answer the following multiple-choice question strictly in **English**.

    First, choose the correct option from A, B, C, or D.  
    Then, provide a detailed explanation or a related scientific lecture that meets the following criteria:

    1. Your explanation must be written entirely in English.  
    2. It should contain at least three complete sentences.  
    3. The explanation must refer to specific visual features in the image.  
    4. Clearly explain your reasoning process step by step.  
    5. Do not use any language other than English.  
    6. Do not just repeat the question or the choices.

    Please follow the format below strictly:

    Answer: <Your choice here, e.g., A>  
    Explanation: <Your detailed explanation or lecture here>
    """
    content.append({"type": "text", "text": question_text})
    
    return [{"role": "user", "content": content}]

# 解析模型输出
import re

def parse_output(output):
    output = output.strip()

    # case 1: "Answer: A Explanation: xxx"
    match = re.search(r"Answer[:：]?\s*([A-D])\b.*?Explanation[:：]?\s*(.+)", output, re.DOTALL)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = match.group(2).strip()
        return answer, explanation

    # case 2: "A Explanation: xxx"
    match = re.match(r"\b([A-D])\s*Explanation[:：]?\s*(.+)", output, re.DOTALL)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = match.group(2).strip()
        return answer, explanation

    # case 3: "A. xxx" or "B: xxx"
    match = re.match(r"\b([A-D])[\.:]\s*(.+)", output, re.DOTALL)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = match.group(2).strip()
        return answer, explanation

    # case 4: only one letter like "C"
    match = re.match(r"^\s*([A-D])\s*$", output)
    if match:
        answer = ord(match.group(1)) - 65
        return answer, ""

    # fallback: try to find first capital letter A-D (unsafe)
    match = re.search(r"\b([A-D])\b", output)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = output[match.end():].strip()
        return answer, explanation

    return -1, ""


# 准备测试数据
N = len(data)
test_dataset = []

for i in range(N):
    sample = data[i]
    try:
        if sample['question'] is None:
            print(f"第 {i} 个样本没有问题，跳过")
            continue

        solution = sample.get("solution", "")
        lecture = sample.get("lecture", "")
        solution_lecture = f"{solution}\n\n{lecture}".strip()

        test_dataset.append({
            "image": sample.get("image", None),
            "question": sample["question"],
            "choices": sample["choices"],
            "answer": sample["answer"],  # 是选项文本
            "hint": sample.get("hint", None),
            "solution_lecture":sample['solution_lecture']
        })
    except Exception as e:
        print(f"跳过第 {i} 个样本，错误：{e}")
        continue

# 初始化工具
rouge = Rouge()
smoothie = SmoothingFunction().method1
vectorizer = CountVectorizer(stop_words="english").fit([s["solution_lecture"] for s in test_dataset])
keywords = set(vectorizer.get_feature_names_out())

# 开始评估
results = []


results = []

for sample in tqdm(test_dataset):
    messages = build_message(sample)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs = [sample["image"]] if sample["image"] else None

    # 构造模型输入
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt", padding=True).to("cuda")

    # 多候选生成
    generated_ids = base_model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        num_return_sequences=3
    )

    # 去除 prompt 部分（仅保留输出）
    prompt_len = inputs.input_ids.shape[1]
    generated_ids_trimmed = [ids[prompt_len:] for ids in generated_ids]

    # 解码
    decoded_outputs = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    # 对每个候选进行打分
    candidate_scores = []
    for out_text in decoded_outputs:
        # 拼接完整输入（同 prompt + image）
        prompt_with_output = text + out_text

        # 重新构建输入以送入 reward model
        rm_inputs = processor(
            text=[prompt_with_output],
            images=image_inputs,
            return_tensors="pt",
            padding=True
        ).to("cuda")

        with torch.no_grad():
            score = rm_model(
                input_ids=rm_inputs["input_ids"],
                attention_mask=rm_inputs["attention_mask"],
                pixel_values=rm_inputs["pixel_values"],
                image_grid_thw=rm_inputs.get("image_grid_thw")  # 如果用了就加上
            ).item()

        candidate_scores.append((score, out_text))

    # 选出得分最高的输出
    best_score, best_output = max(candidate_scores, key=lambda x: x[0])

    results.append({
        "question": sample["question"],
        "candidates": [x[1] for x in candidate_scores],
        "scores": [x[0] for x in candidate_scores],
        "best_output": best_output,
        "best_score": best_score,
        "lecture": sample["solution_lecture"]
    })
        # 解析最佳输出答案
    pred_answer_idx, pred_explanation = parse_output(best_output)

    # ground truth 答案是文本（例如 "The Earth rotates..."）
    # 找到选项索引（0=A, 1=B, ...）进行准确率评估
    try:
        true_answer_idx = sample["choices"].index(sample["answer"])
    except ValueError:
        true_answer_idx = -1

    # BLEU-1 和 BLEU-4
    reference = sample["solution_lecture"].split()
    hypothesis = pred_explanation.split()
    bleu1 = sentence_bleu([reference], hypothesis, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu4 = sentence_bleu([reference], hypothesis, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    # ROUGE-L
    try:
        rouge_score = rouge.get_scores(pred_explanation, sample["solution_lecture"])[0]["rouge-l"]["f"]
    except:
        rouge_score = 0.0

    # 关键词重叠
    gt_tokens = set(sample["solution_lecture"].lower().split())
    pred_tokens = set(pred_explanation.lower().split())
    overlap = len(gt_tokens & pred_tokens & keywords)
    keyword_score = overlap / max(len(gt_tokens & keywords), 1)

    # 加入结果
    results[-1].update({
        "predicted_answer_idx": pred_answer_idx,
        "predicted_answer_text": sample["choices"][pred_answer_idx] if 0 <= pred_answer_idx < len(sample["choices"]) else "N/A",
        "true_answer_idx": true_answer_idx,
        "bleu1": bleu1,
        "bleu4": bleu4,
        "rougeL": rouge_score,
        "keyword_overlap": keyword_score,
        "correct": int(pred_answer_idx == true_answer_idx)
    })

    # 将结果列表转换为 DataFrame
df = pd.DataFrame(results)

# 如果存在无效预测（-1），可选择过滤掉：
valid_df = df[df["predicted_answer_idx"] >= 0].copy()

# 计算整体指标
acc = accuracy_score(valid_df["true_answer_idx"], valid_df["predicted_answer_idx"])
f1 = f1_score(valid_df["true_answer_idx"], valid_df["predicted_answer_idx"], average="macro")
avg_bleu1 = valid_df["bleu1"].mean()
avg_bleu4 = valid_df["bleu4"].mean()
avg_rougeL = valid_df["rougeL"].mean()
avg_keyword_overlap = valid_df["keyword_overlap"].mean()

# 打印结果
print("\n📊 Overall Evaluation Metrics:")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score (macro): {f1:.4f}")
print(f"Avg BLEU-1: {avg_bleu1:.4f}")
print(f"Avg BLEU-4: {avg_bleu4:.4f}")
print(f"Avg ROUGE-L: {avg_rougeL:.4f}")
print(f"Avg Keyword Overlap: {avg_keyword_overlap:.4f}")



100%|██████████| 1429/1429 [2:54:24<00:00,  7.32s/it]  


📊 Overall Evaluation Metrics:
Accuracy: 0.0000
F1 Score (macro): 0.0000
Avg BLEU-1: 0.4603
Avg BLEU-4: 0.3568
Avg ROUGE-L: 0.5518
Avg Keyword Overlap: 0.5186


In [7]:
results

[{'question': 'What is the name of the colony shown?',
  'candidates': ['Answer: B\nExplanation: The colony is New Hampshire.',
   'Answer: B\nExplanation: The colony is New Hampshire.',
   'Answer: B\nExplanation: The colony is New Hampshire.'],
  'scores': [0.5390625, 0.5390625, 0.5390625],
  'best_output': 'Answer: B\nExplanation: The colony is New Hampshire.',
  'best_score': 0.5390625,
  'lecture': 'The colony is New Hampshire.\nDuring the colonial era, New Hampshire and New York both claimed the territory that would later become the state of Vermont. Vermont was never its own colony.',
  'predicted_answer_idx': 1,
  'predicted_answer_text': 'New Hampshire',
  'true_answer_idx': -1,
  'bleu1': 0.004516580942612666,
  'bleu4': 0.004516580942612666,
  'rougeL': 0.33333333055555564,
  'keyword_overlap': 0.2,
  'correct': 0},
 {'question': 'Which of these organisms contains matter that was once part of the lichen?',
  'candidates': ['Answer: B \nExplanation: \nIn a food web, each arro